<a href="https://colab.research.google.com/github/navathipravallika-std/Myfirstagent/blob/main/MyFirstagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

%pip install -q -U langchain langchain-google-genai langgraph


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [1]:
%pip install google-auth==2.49.0

In [3]:

# ==========================================================
# LangChain for AI Agents — Companion Code
# Google Colab + Gemini
# ==========================================================
# Demonstrates:
# 1. ReAct: Reason -> Act -> Observe
# 2. Four math tools
# 3. Agent creation with create_agent()
# 4. Execution trace and test cases
# ==========================================================

import os
import math

from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool


# -------------------------------------------------------
# STEP 0: Load Gemini API key
# -------------------------------------------------------

api_key = userdata.get("GEMINI_API_KEY")

if not api_key:
    raise ValueError(
        "GEMINI_API_KEY is missing. Add it in Colab Secrets "
        "and enable notebook access."
    )

os.environ["GOOGLE_API_KEY"] = api_key

print("Gemini API key loaded successfully!")


# -------------------------------------------------------
# STEP 1: Initialize the model
# -------------------------------------------------------

# If this model ID is unavailable, select a current model
# from Google AI Studio that supports function/tool calling.

MODEL_NAME = "gemini-3.8-flash"

model = init_chat_model(
    f"google_genai:{MODEL_NAME}"
)

print(f"Model initialized: {MODEL_NAME}")


# -------------------------------------------------------
# STEP 2: Define the tools
# -------------------------------------------------------

@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b


@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b


@tool
def divide(a: float, b: float) -> str:
    """Divide the first number by the second.
    Return an error message if the second number is zero.
    """
    if b == 0:
        return "Error: Cannot divide by zero."
    return str(a / b)


@tool
def square_root(number: float) -> str:
    """Calculate the square root of a non-negative number."""
    if number < 0:
        return "Error: Cannot take the square root of a negative number."
    return str(math.sqrt(number))


# IMPORTANT: Create the tools list before creating the agent.
tools = [add, multiply, divide, square_root]

print("\n=== Available Tools ===")
for t in tools:
    print(f"- {t.name}: {t.description}")


# -------------------------------------------------------
# STEP 3: Create the agent
# -------------------------------------------------------

agent = create_agent(
    model=model,
    tools=tools
)

print("\nAI Agent created successfully!")


# -------------------------------------------------------
# STEP 4: Run the agent and display its execution trace
# -------------------------------------------------------

def run_agent(question: str):
    """Run the agent and display tool calls and results."""

    print(f"\nUser: {question}")
    print("-" * 60)

    result = agent.invoke({
        "messages": [("user", question)]
    })

    print("AGENT EXECUTION TRACE")
    print("-" * 60)

    step = 1

    for msg in result["messages"]:

        # Original user question
        if msg.type == "human":
            print(f"{step}. User asked:")
            print(f"   {msg.content}")
            step += 1

        # Agent tool-call decisions
        elif msg.type == "ai" and getattr(msg, "tool_calls", None):
            for call in msg.tool_calls:
                print(f"{step}. Agent decision:")
                print(f"   Tool: {call['name']}")
                print(f"   Input: {call['args']}")
                step += 1

        # Tool results
        elif msg.type == "tool":
            print(f"{step}. Tool observation:")
            print(f"   Result: {msg.content}")
            step += 1

        # Final answer: only print AI messages without tool calls
        elif msg.type == "ai" and msg.content:
            print(f"{step}. Final answer:")
            print(f"   {msg.content}")
            step += 1

    print("=" * 60)


# -------------------------------------------------------
# STEP 5: Test cases
# -------------------------------------------------------

# Test 1: Simple calculation
run_agent("What is 42 + 58?")

# Test 2: Multiple operations
run_agent("What is 15 multiplied by 8, then divided by 3?")

# Test 3: Multiple tool calls
run_agent(
    "A rectangle has width 12 and height 7. "
    "Calculate its area and then the square root of its area."
)

# Test 4: Division by zero
run_agent("What is 100 divided by 0?")

print("\nAI Agent demo complete!")


Gemini API key loaded successfully!
Model initialized: gemini-3.8-flash

=== Available Tools ===
- add: Add two numbers together.
- multiply: Multiply two numbers together.
- divide: Divide the first number by the second.
Return an error message if the second number is zero.
- square_root: Calculate the square root of a non-negative number.

AI Agent created successfully!

User: What is 42 + 58?
------------------------------------------------------------
AGENT EXECUTION TRACE
------------------------------------------------------------
1. User asked:
   What is 42 + 58?
2. Agent decision:
   Tool: add
   Input: {'b': 58, 'a': 42}
3. Tool observation:
   Result: 100.0
4. Final answer:
   [{'type': 'text', 'text': '42 + 58 = 100.', 'extras': {'signature': 'EnsKeQFpFH0T2xaB/M3tXfG6rypsUJhWSRnaPWxi5w3oabPamb/aPKCs2EFXxJ77LvBlVEFpTe5nO3SQBbwUXb1mQOrZl66D2mG29+E2I/sNy7rtKvadxXDaAbwciB5RXJhLy5EQM1zSnIHsb0XEz0NHu9mQsOjAEJeqHTY='}}]

User: What is 15 multiplied by 8, then divided by 3?
-------

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 1h32m6.855886381s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'G